In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings, os
 
warnings.filterwarnings("ignore")
 
# ─── Paramètres de style ────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
COLORS = {"Douala": "#E63946", "Yaoundé": "#457B9D", "Bafoussam": "#2A9D8F"}
PALETTE = list(COLORS.values())
 
OUT = r"C:\Users\AMOS SARL\OneDrive\Documents\Projet_1\outputs"
os.makedirs(OUT, exist_ok=True) 
from scipy import stats
import warnings, os
 
warnings.filterwarnings("ignore")
 
# ─── Paramètres de style ────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
COLORS = {"Douala": "#E63946", "Yaoundé": "#457B9D", "Bafoussam": "#2A9D8F"}
PALETTE = list(COLORS.values())
 
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)


# 0. CHARGEMENT ET NETTOYAGE MINIMAL (pré-requis Semaine 3)
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("0. CHARGEMENT ET NETTOYAGE MINIMAL DES DONNÉES")
print("=" * 60)
 
orders_raw = pd.read_csv(r"C:/Users/AMOS SARL/OneDrive/Documents/Projet_1/Nexa_commerce/data/orders.csv")
 
# Normalisation ville
def normalize_city(c):
    c = str(c).strip().lower()
    if "douala" in c:   return "Douala"
    if "yaound" in c:   return "Yaoundé"
    if "bafou"  in c:   return "Bafoussam"
    return "Inconnu"
 
# Normalisation statut
def normalize_status(s):
    s = str(s).strip().lower().replace(" ", "_")
    if s.startswith("livr"):   return "livré"
    if "retard" in s:          return "en_retard"
    if "annul" in s:           return "annulé"
    if "cours" in s:           return "en_cours"
    return s
 
orders = orders_raw.copy()
orders["city"]   = orders["city"].apply(normalize_city)
orders["status"] = orders["status"].apply(normalize_status)
 
# Parse dates – plusieurs formats
def parse_date(d):
    for fmt in ["%d/%m/%Y %H:%M", "%Y-%m-%d %H:%M:%S", "%d-%m-%Y", "%Y-%m-%d"]:
        try:
            return pd.to_datetime(d, format=fmt)
        except:
            pass
    return pd.NaT
 
orders["order_dt"]   = orders["order_date"].apply(parse_date)
orders["order_hour"] = orders["order_dt"].dt.hour
orders["day_of_week"] = orders["order_dt"].dt.dayofweek   # 0=lundi, 4=vendredi
# Exclure montants négatifs et delivery_time_min abérrante
orders = orders[orders["total_amount_xaf"] > 0]
df = orders.dropna(subset=["delivery_time_min"]).copy()
df = df[df["delivery_time_min"] > 0]
 
print(f"Commandes après nettoyage minimal : {len(df):,}")
print(f"Villes : {df['city'].value_counts().to_dict()}")
print(f"Statuts : {df['status'].value_counts().to_dict()}\n")



0. CHARGEMENT ET NETTOYAGE MINIMAL DES DONNÉES
Commandes après nettoyage minimal : 10,122
Villes : {'Douala': 5208, 'Yaoundé': 3349, 'Bafoussam': 1565}
Statuts : {'livré': 6769, 'en_retard': 2361, 'annulé': 992}



In [17]:
# MISSION 1 – STATISTIQUES DESCRIPTIVES PAR VILLE
print("=" * 60)
print("MISSION 1 – STATISTIQUES DESCRIPTIVES DES TEMPS DE LIVRAISON")
print("=" * 60)
 
stats_desc = {}
for city in ["Douala", "Yaoundé", "Bafoussam"]:
    data = df[df["city"] == city]["delivery_time_min"]
    stats_desc[city] = {
        "n":          len(data),
        "moyenne":    round(data.mean(), 2),
        "médiane":    round(data.median(), 2),
        "écart-type": round(data.std(), 2),
        "Q1":         round(data.quantile(0.25), 2),
        "Q3":         round(data.quantile(0.75), 2),
        "IQR":        round(data.quantile(0.75) - data.quantile(0.25), 2),
        "min":        round(data.min(), 2),
        "max":        round(data.max(), 2),
    }
 
stats_df = pd.DataFrame(stats_desc).T
print(stats_df.to_string())
print()

# ── Figure 1 : Histogrammes + KDE ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
fig.suptitle("Distribution des temps de livraison par ville\n(NexaCommerce – 2023-2025)",
             fontsize=14, fontweight="bold", y=1.02)
 
for ax, city in zip(axes, ["Douala", "Yaoundé", "Bafoussam"]):
    data  = df[df["city"] == city]["delivery_time_min"]
    color = COLORS[city]
    ax.hist(data, bins=35, color=color, alpha=0.65, density=True, edgecolor="white", linewidth=0.4)
    kde_x = np.linspace(data.min(), data.max(), 300)
    kde   = stats.gaussian_kde(data)(kde_x)
    ax.plot(kde_x, kde, color=color, linewidth=2.5)
    ax.axvline(data.mean(),   color="black",  linestyle="--", linewidth=1.4, label=f"Moyenne {data.mean():.0f} min")
    ax.axvline(data.median(), color="orange", linestyle=":",  linewidth=1.6, label=f"Médiane {data.median():.0f} min")
    ax.set_title(city, color=color, fontweight="bold")
    ax.set_xlabel("Temps de livraison (min)")
    ax.set_ylabel("Densité")
    ax.legend(fontsize=8)
 
plt.tight_layout()
plt.savefig(f"{OUT}/S3_M1_histogrammes.png", bbox_inches="tight")
plt.close()
print("→ Figure 1 sauvegardée : S3_M1_histogrammes.png")

# ── Figure 2 : Boxplots comparatifs ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
data_by_city = [df[df["city"] == c]["delivery_time_min"].values for c in ["Douala", "Yaoundé", "Bafoussam"]]
bp = ax.boxplot(data_by_city, patch_artist=True, notch=True,
                medianprops={"color": "white", "linewidth": 2})
for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_xticklabels(["Douala", "Yaoundé", "Bafoussam"])
ax.set_xlabel("Ville de livraison")
ax.set_ylabel("Temps de livraison (minutes)")
ax.set_title("Boxplots des temps de livraison par ville\n(encoches = IC 95% de la médiane)")
ax.axhline(df["delivery_time_min"].mean(), color="gray", linestyle="--", alpha=0.5, label="Moyenne globale")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/S3_M1_boxplots.png", bbox_inches="tight")
plt.close()
print("→ Figure 2 sauvegardée : S3_M1_boxplots.png\n")

# Interprétation
for city, s in stats_desc.items():
    skew = "distribution asymétrique droite" if s["moyenne"] > s["médiane"] else "distribution symétrique"
    print(f"  {city} : moyenne={s['moyenne']} min, médiane={s['médiane']} min → {skew}")
print()



MISSION 1 – STATISTIQUES DESCRIPTIVES DES TEMPS DE LIVRAISON
                n  moyenne  médiane  écart-type    Q1     Q3    IQR  min    max
Douala     5208.0    63.98     43.2       53.56  31.5  77.93  46.43  5.0  338.9
Yaoundé    3349.0    78.74     53.2       64.98  38.4  99.20  60.80  5.0  417.8
Bafoussam  1565.0    51.81     36.0       41.60  25.8  63.60  37.80  5.0  263.4

→ Figure 1 sauvegardée : S3_M1_histogrammes.png
→ Figure 2 sauvegardée : S3_M1_boxplots.png

  Douala : moyenne=63.98 min, médiane=43.2 min → distribution asymétrique droite
  Yaoundé : moyenne=78.74 min, médiane=53.2 min → distribution asymétrique droite
  Bafoussam : moyenne=51.81 min, médiane=36.0 min → distribution asymétrique droite



In [16]:
# MISSION 2 – THÉORÈME DE BAYES
# P(retard | vendredi soir 18h-23h, Douala)
print("=" * 60)
print("MISSION 2 – THÉORÈME DE BAYES")
print("P(retard | vendredi soir à Douala)")
print("=" * 60)
print("""
Formule de Bayes :
P(retard | vendredi_soir_Douala) = P(vendredi_soir_Douala | retard) × P(retard)
                                    ─────────────────────────────────────────────
                                              P(vendredi_soir_Douala)
 
Définitions :
  - "vendredi soir"  : commandes passées un vendredi (dayofweek=4) entre 18h et 23h
  - "retard"         : statut == 'en_retard'
  - "Douala"         : city == 'Douala'
  - Événement A      : retard
  - Événement B      : vendredi soir à Douala
""")
# Sous-ensemble pertinent (commandes Douala avec date parsée)
df_d = df[df["city"] == "Douala"].copy()
n_total = len(df_d)
 
# Événement B : vendredi (4) soir (18h-23h)
mask_vs = (df_d["day_of_week"] == 4) & (df_d["order_hour"].between(18, 23))
n_B     = mask_vs.sum()
 
# Événement A : retard (dans Douala)
mask_R  = df_d["status"] == "en_retard"
n_A     = mask_R.sum()
 
# A ∩ B
n_AB    = (mask_vs & mask_R).sum()
 
P_A     = n_A  / n_total        # P(retard | Douala)
P_B     = n_B  / n_total        # P(vendredi soir | Douala)
P_B_given_A = n_AB / n_A        # P(vendredi soir | retard, Douala)
P_A_given_B = (P_B_given_A * P_A) / P_B  # Bayes
 
print(f"  Commandes Douala                    : {n_total:,}")
print(f"  Commandes en retard (A)             : {n_A:,}")
print(f"  Commandes vendredi soir (B)         : {n_B:,}")
print(f"  Commandes retard ∩ vendredi soir    : {n_AB:,}")
print()
print(f"  P(retard | Douala)                  = {P_A:.4f}  ({P_A*100:.1f}%)")
print(f"  P(vendredi_soir | Douala)           = {P_B:.4f}  ({P_B*100:.1f}%)")
print(f"  P(vendredi_soir | retard, Douala)   = {P_B_given_A:.4f}  ({P_B_given_A*100:.1f}%)")
print()
print(f"  ┌─────────────────────────────────────────────────┐")
print(f"  │ P(retard | vendredi soir, Douala) = {P_A_given_B:.4f}       │")
print(f"  │                                   = {P_A_given_B*100:.1f}%         │")
print(f"  └─────────────────────────────────────────────────┘")
print()
# Vérification directe (fréquentiste)
p_direct = n_AB / n_B
print(f"  Vérification fréquentiste directe  = {p_direct:.4f}  ({p_direct*100:.1f}%) ✓")
 
# Comparaison avec taux de base
print()
risk_ratio = P_A_given_B / P_A
print(f"  → Un vendredi soir, le risque de retard est {risk_ratio:.1f}x")
print(f"    plus élevé que le taux moyen à Douala ({P_A*100:.1f}%).")
print()
 
# Figure Bayes
fig, ax = plt.subplots(figsize=(8, 5))
labels   = ["Taux de retard\nglobal Douala", "Taux de retard\nvendredi soir Douala"]
values   = [P_A * 100, P_A_given_B * 100]
bar_colors = ["#457B9D", "#E63946"]
bars = ax.bar(labels, values, color=bar_colors, width=0.4, edgecolor="white")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=12)
ax.set_ylabel("Probabilité de retard (%)")
ax.set_title("Théorème de Bayes\nImpact du créneau « vendredi soir » sur le risque de retard à Douala")
ax.set_ylim(0, max(values) * 1.25)
plt.tight_layout()
plt.savefig(f"{OUT}/S3_M2_bayes.png", bbox_inches="tight")
plt.close()
print("→ Figure 3 sauvegardée : S3_M2_bayes.png\n")





MISSION 2 – THÉORÈME DE BAYES
P(retard | vendredi soir à Douala)

Formule de Bayes :
P(retard | vendredi_soir_Douala) = P(vendredi_soir_Douala | retard) × P(retard)
                                    ─────────────────────────────────────────────
                                              P(vendredi_soir_Douala)

Définitions :
  - "vendredi soir"  : commandes passées un vendredi (dayofweek=4) entre 18h et 23h
  - "retard"         : statut == 'en_retard'
  - "Douala"         : city == 'Douala'
  - Événement A      : retard
  - Événement B      : vendredi soir à Douala

  Commandes Douala                    : 5,208
  Commandes en retard (A)             : 1,169
  Commandes vendredi soir (B)         : 188
  Commandes retard ∩ vendredi soir    : 47

  P(retard | Douala)                  = 0.2245  (22.4%)
  P(vendredi_soir | Douala)           = 0.0361  (3.6%)
  P(vendredi_soir | retard, Douala)   = 0.0402  (4.0%)

  ┌─────────────────────────────────────────────────┐
  │ P(retard | vendre

In [13]:
# MISSION 3 – TEST DE STUDENT (t-test)
# Douala vs Yaoundé : différence significative des temps de livraison ?
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("MISSION 3 – TEST DE STUDENT (t-test bilatéral)")
print("H₀ : μ_Douala = μ_Yaoundé   |   H₁ : μ_Douala ≠ μ_Yaoundé")
print("α = 0,05")
print("=" * 60)
 
data_douala  = df[df["city"] == "Douala"]["delivery_time_min"].dropna()
data_yaounde = df[df["city"] == "Yaoundé"]["delivery_time_min"].dropna()
 
print(f"\n  Douala   : n={len(data_douala):,}, μ={data_douala.mean():.2f} min, σ={data_douala.std():.2f}")
print(f"  Yaoundé  : n={len(data_yaounde):,}, μ={data_yaounde.mean():.2f} min, σ={data_yaounde.std():.2f}")
 
# Test de Levene (égalité des variances)
lev_stat, lev_p = stats.levene(data_douala, data_yaounde)
equal_var = lev_p > 0.05
print(f"\n  Test de Levene (égalité variances) : F={lev_stat:.3f}, p={lev_p:.4f}")
print(f"  → Variances {'égales' if equal_var else 'inégales'} (Welch t-test {'non ' if equal_var else ''}recommandé)")
 
# t-test de Welch (robuste même si variances inégales)
t_stat, p_value = stats.ttest_ind(data_douala, data_yaounde, equal_var=False)
 
# Degrés de liberté approximatifs (Welch–Satterthwaite)
n1, s1 = len(data_douala),  data_douala.std()
n2, s2 = len(data_yaounde), data_yaounde.std()
df_welch = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))
t_crit   = stats.t.ppf(0.975, df_welch)
 
# Taille d'effet – Cohen's d
pooled_std = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1 + n2 - 2))
cohens_d   = (data_douala.mean() - data_yaounde.mean()) / pooled_std
 
print(f"\n  t de Welch    = {t_stat:.4f}")
print(f"  Valeur p      = {p_value:.6f}")
print(f"  t critique    = ±{t_crit:.4f}  (α/2=0,025, ddl≈{df_welch:.0f})")
print(f"  Cohen's d     = {cohens_d:.4f}")
print()
 
alpha = 0.05
if p_value < alpha:
    print(f"  ┌────────────────────────────────────────────────────────┐")
    print(f"  │ p={p_value:.4f} < α=0,05 → On REJETTE H₀               │")
    print(f"  │ La différence est STATISTIQUEMENT SIGNIFICATIVE.        │")
    print(f"  │ Yaoundé livre {data_yaounde.mean()-data_douala.mean():.1f} min plus lentement que Douala. │")
    print(f"  └────────────────────────────────────────────────────────┘")
else:
    print(f"  p={p_value:.4f} ≥ α=0,05 → On ne rejette pas H₀")
    print(f"  Pas de différence significative.")
 
d_interp = "faible" if abs(cohens_d)<0.2 else "modéré" if abs(cohens_d)<0.5 else "fort" if abs(cohens_d)<0.8 else "très fort"
print(f"\n  Taille d'effet : Cohen's d = {cohens_d:.3f} → effet {d_interp}")
 
# Intervalle de confiance à 95% sur la différence
diff    = data_douala.mean() - data_yaounde.mean()
se_diff = np.sqrt(s1**2/n1 + s2**2/n2)
ci_lo   = diff - t_crit * se_diff
ci_hi   = diff + t_crit * se_diff
print(f"\n  IC 95% sur la différence μ_D - μ_Y : [{ci_lo:.2f}, {ci_hi:.2f}] min")
print()
 
# Figure t-test
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
# Gauche : distributions superposées
ax = axes[0]
for city, data, color in [("Douala", data_douala, COLORS["Douala"]),
                           ("Yaoundé", data_yaounde, COLORS["Yaoundé"])]:
    ax.hist(data, bins=30, density=True, alpha=0.45, color=color, label=city)
    kde_x = np.linspace(data.min(), data.max(), 300)
    ax.plot(kde_x, stats.gaussian_kde(data)(kde_x), color=color, linewidth=2)
    ax.axvline(data.mean(), color=color, linestyle="--", linewidth=1.5)
ax.set_xlabel("Temps de livraison (min)")
ax.set_ylabel("Densité")
ax.set_title("Comparaison des distributions\nDouala vs Yaoundé")
ax.legend()
 
# Droite : distribution t avec zones de rejet
ax2 = axes[1]
x   = np.linspace(-5, 5, 500)
ax2.plot(x, stats.t.pdf(x, df_welch), color="steelblue", linewidth=2)
ax2.fill_between(x, stats.t.pdf(x, df_welch), where=(x < -t_crit), color="#E63946", alpha=0.4, label=f"Rejet H₀ (α/2)")
ax2.fill_between(x, stats.t.pdf(x, df_welch), where=(x >  t_crit), color="#E63946", alpha=0.4)
ax2.axvline(t_stat, color="black", linewidth=2.5, linestyle="-", label=f"t observé = {t_stat:.2f}")
ax2.axvline(-t_crit, color="#E63946", linewidth=1.5, linestyle="--")
ax2.axvline( t_crit, color="#E63946", linewidth=1.5, linestyle="--", label=f"t critique = ±{t_crit:.2f}")
ax2.set_xlabel("t")
ax2.set_ylabel("Densité")
ax2.set_title(f"Distribution t de Student\n(p = {p_value:.4f})")
ax2.legend(fontsize=9)
 
plt.suptitle("Test t de Welch : temps de livraison Douala vs Yaoundé", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/S3_M3_ttest.png", bbox_inches="tight")
plt.close()
print("→ Figure 4 sauvegardée : S3_M3_ttest.png\n")


MISSION 3 – TEST DE STUDENT (t-test bilatéral)
H₀ : μ_Douala = μ_Yaoundé   |   H₁ : μ_Douala ≠ μ_Yaoundé
α = 0,05

  Douala   : n=5,208, μ=63.98 min, σ=53.56
  Yaoundé  : n=3,349, μ=78.74 min, σ=64.98

  Test de Levene (égalité variances) : F=62.877, p=0.0000
  → Variances inégales (Welch t-test recommandé)

  t de Welch    = -10.9676
  Valeur p      = 0.000000
  t critique    = ±1.9603  (α/2=0,025, ddl≈6157)
  Cohen's d     = -0.2532

  ┌────────────────────────────────────────────────────────┐
  │ p=0.0000 < α=0,05 → On REJETTE H₀               │
  │ La différence est STATISTIQUEMENT SIGNIFICATIVE.        │
  │ Yaoundé livre 14.8 min plus lentement que Douala. │
  └────────────────────────────────────────────────────────┘

  Taille d'effet : Cohen's d = -0.253 → effet modéré

  IC 95% sur la différence μ_D - μ_Y : [-17.40, -12.12] min

→ Figure 4 sauvegardée : S3_M3_ttest.png



In [14]:
# MISSION 4 – MATRICE DE CORRÉLATION
# Variables : montant, delivery_time, loyalty_score, heure de commande

print("=" * 60)
print("MISSION 4 – MATRICE DE CORRÉLATION")
print("=" * 60)
 
# Joindre customers pour loyalty_score
customers_raw = pd.read_csv(r"C:/Users/AMOS SARL/OneDrive/Documents/Projet_1/Nexa_commerce/data/customers.csv")
customers_raw["loyalty_score"] = customers_raw["loyalty_score"].clip(upper=100)  # corriger aberrants > 100
 
df_corr = df.merge(customers_raw[["customer_id","loyalty_score"]],
                   on="customer_id", how="left")
df_corr = df_corr.dropna(subset=["loyalty_score","delivery_time_min","total_amount_xaf"])
 
corr_vars = df_corr[["total_amount_xaf", "delivery_time_min", "loyalty_score", "order_hour"]].copy()
corr_vars.columns = ["Montant (XAF)", "Temps livraison (min)", "Score fidélité", "Heure commande"]
 
corr_matrix = corr_vars.corr(method="pearson")
print("Matrice de corrélation de Pearson :")
print(corr_matrix.round(4).to_string())
print()
 
# Tests de significativité
# Nettoyage : on enlève les lignes où une des 4 variables est manquante
corr_vars_clean = corr_vars.dropna()   # <- TOUS les NaN supprimés

# Matrice de corrélation (sur données propres)
corr_matrix = corr_vars_clean.corr(method="pearson")
print("Matrice de corrélation de Pearson :")
print(corr_matrix.round(4).to_string())

# Tests de significativité (les longueurs sont identiques)
print("\nTests de significativité (p-values) :")
cols = corr_vars_clean.columns
pval_matrix = pd.DataFrame(np.ones((4,4)), index=cols, columns=cols)
for i, c1 in enumerate(cols):
    for j, c2 in enumerate(cols):
        if i != j:
            r, p = stats.pearsonr(corr_vars_clean[c1], corr_vars_clean[c2])
            pval_matrix.loc[c1, c2] = p
print(pval_matrix.round(4).to_string())
 
# Figure matrice de corrélation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
# Heatmap
ax = axes[0]
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, mask=mask,
            linewidths=0.5, ax=ax, square=True,
            annot_kws={"size": 10, "weight": "bold"})
ax.set_title("Matrice de corrélation de Pearson\n(triangle inférieur)")
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)
 
# Scatter : heure vs delivery_time
ax2 = axes[1]
for city, color in COLORS.items():
    mask_c = df_corr["city"] == city
    ax2.scatter(df_corr.loc[mask_c, "order_hour"],
                df_corr.loc[mask_c, "delivery_time_min"],
                alpha=0.15, s=12, color=color, label=city)
# Régression globale
m, b, r, p, se = stats.linregress(df_corr["order_hour"], df_corr["delivery_time_min"])
x_line = np.linspace(6, 23, 100)
ax2.plot(x_line, m*x_line + b, color="black", linewidth=2,
         label=f"y = {m:.2f}x + {b:.1f}  (r={r:.3f})")
ax2.set_xlabel("Heure de la commande")
ax2.set_ylabel("Temps de livraison (min)")
ax2.set_title("Heure de commande vs Temps de livraison")
ax2.legend(fontsize=8)
 
plt.suptitle("Analyse de corrélation – NexaCommerce", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/S3_M4_correlation.png", bbox_inches="tight")
plt.close()
print("→ Figure 5 sauvegardée : S3_M4_correlation.png\n")


MISSION 4 – MATRICE DE CORRÉLATION
Matrice de corrélation de Pearson :
                       Montant (XAF)  Temps livraison (min)  Score fidélité  Heure commande
Montant (XAF)                 1.0000                 0.0022          0.0029         -0.0139
Temps livraison (min)         0.0022                 1.0000         -0.0126          0.0083
Score fidélité                0.0029                -0.0126          1.0000          0.0096
Heure commande               -0.0139                 0.0083          0.0096          1.0000

Matrice de corrélation de Pearson :
                       Montant (XAF)  Temps livraison (min)  Score fidélité  Heure commande
Montant (XAF)                 1.0000                 0.0040         -0.0017         -0.0139
Temps livraison (min)         0.0040                 1.0000         -0.0207          0.0083
Score fidélité               -0.0017                -0.0207          1.0000          0.0096
Heure commande               -0.0139                 0.0083     

In [15]:
# MISSION 5 – DÉTECTION D'OUTLIERS PAR MÉTHODE IQR
# ═══════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("MISSION 5 – DÉTECTION D'OUTLIERS (méthode IQR)")
print("=" * 60)
print("""
Justification mathématique :
  Q1 = 25ème percentile, Q3 = 75ème percentile
  IQR = Q3 - Q1  (dispersion inter-quartile)
 
  Seuil bas  : Q1 - 1.5 × IQR  → valeurs anormalement courtes
  Seuil haut : Q3 + 1.5 × IQR  → valeurs anormalement longues
 
  Le facteur 1.5 provient de la règle de Tukey (1977).
  Pour une distribution normale :
    - 99.3% des données se trouvent dans [Q1-1.5IQR, Q3+1.5IQR]
    - Seul 0.7% sont théoriquement outliers → seuil conservateur
  Facteur 3.0 = "outliers extrêmes" (> 1 chance sur 2000 pour loi normale)
""")
 
results_iqr = {}
for city in ["Douala", "Yaoundé", "Bafoussam", "GLOBAL"]:
    if city == "GLOBAL":
        data = df["delivery_time_min"]
    else:
        data = df[df["city"] == city]["delivery_time_min"]
 
    Q1  = data.quantile(0.25)
    Q3  = data.quantile(0.75)
    IQR = Q3 - Q1
 
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
 
    # Outliers extrêmes (Tukey, k=3)
    lower_ext = Q1 - 3.0 * IQR
    upper_ext = Q3 + 3.0 * IQR
 
    outliers_low   = data[data < lower]
    outliers_high  = data[data > upper]
    outliers_ext   = data[(data < lower_ext) | (data > upper_ext)]
 
    results_iqr[city] = {
        "Q1": Q1, "Q3": Q3, "IQR": IQR,
        "Seuil bas (k=1.5)":  lower,
        "Seuil haut (k=1.5)": upper,
        "n_outliers_bas":      len(outliers_low),
        "n_outliers_haut":     len(outliers_high),
        "n_outliers_extrêmes": len(outliers_ext),
        "% outliers":          round((len(outliers_low)+len(outliers_high))/len(data)*100, 2),
    }
 
    print(f"  [{city}]")
    print(f"    Q1={Q1:.1f} min | Q3={Q3:.1f} min | IQR={IQR:.1f} min")
    print(f"    Seuil bas  = {lower:.1f} min | Seuil haut = {upper:.1f} min")
    print(f"    Outliers bas : {len(outliers_low)} | Outliers hauts : {len(outliers_high)}")
    print(f"    % total : {results_iqr[city]['% outliers']}%")
    print()
 
# Figure IQR
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Détection d'outliers par méthode IQR (Tukey)\nTemps de livraison – NexaCommerce",
             fontweight="bold", fontsize=13)
 
city_list = ["Douala", "Yaoundé", "Bafoussam", "GLOBAL"]
ax_list   = axes.flatten()
 
for ax, city in zip(ax_list, city_list):
    if city == "GLOBAL":
        data  = df["delivery_time_min"]
        color = "#6A0572"
        label = "Global"
    else:
        data  = df[df["city"] == city]["delivery_time_min"]
        color = COLORS[city]
        label = city
 
    r   = results_iqr[city]
    lo  = r["Seuil bas (k=1.5)"]
    hi  = r["Seuil haut (k=1.5)"]
 
    is_normal  = (data >= lo) & (data <= hi)
    is_outlier = ~is_normal
 
    ax.scatter(range(len(data[is_normal])),
               sorted(data[is_normal].values), s=4, alpha=0.3, color=color, label="Normal")
    ax.scatter(range(len(data[is_outlier])),
               sorted(data[is_outlier].values), s=15, alpha=0.8, color="red",
               marker="x", label=f"Outliers ({is_outlier.sum()})")
    ax.axhline(lo, color="orange", linewidth=1.5, linestyle="--", label=f"Seuil bas {lo:.0f}")
    ax.axhline(hi, color="orange", linewidth=1.5, linestyle="-.",  label=f"Seuil haut {hi:.0f}")
    ax.set_title(f"{label}  ({r['% outliers']}% outliers)")
    ax.set_xlabel("Index")
    ax.set_ylabel("Temps de livraison (min)")
    ax.legend(fontsize=7)
 
plt.tight_layout()
plt.savefig(f"{OUT}/S3_M5_outliers_iqr.png", bbox_inches="tight")
plt.close()
print("→ Figure 6 sauvegardée : S3_M5_outliers_iqr.png")







MISSION 5 – DÉTECTION D'OUTLIERS (méthode IQR)

Justification mathématique :
  Q1 = 25ème percentile, Q3 = 75ème percentile
  IQR = Q3 - Q1  (dispersion inter-quartile)

  Seuil bas  : Q1 - 1.5 × IQR  → valeurs anormalement courtes
  Seuil haut : Q3 + 1.5 × IQR  → valeurs anormalement longues

  Le facteur 1.5 provient de la règle de Tukey (1977).
  Pour une distribution normale :
    - 99.3% des données se trouvent dans [Q1-1.5IQR, Q3+1.5IQR]
    - Seul 0.7% sont théoriquement outliers → seuil conservateur
  Facteur 3.0 = "outliers extrêmes" (> 1 chance sur 2000 pour loi normale)

  [Douala]
    Q1=31.5 min | Q3=77.9 min | IQR=46.4 min
    Seuil bas  = -38.1 min | Seuil haut = 147.6 min
    Outliers bas : 0 | Outliers hauts : 425
    % total : 8.16%

  [Yaoundé]
    Q1=38.4 min | Q3=99.2 min | IQR=60.8 min
    Seuil bas  = -52.8 min | Seuil haut = 190.4 min
    Outliers bas : 0 | Outliers hauts : 203
    % total : 6.06%

  [Bafoussam]
    Q1=25.8 min | Q3=63.6 min | IQR=37.8 min
    S

In [12]:
# Résumé Semaine 3
print()
print("=" * 60)
print("RÉSUMÉ SEMAINE 3")
print("=" * 60)
print(f"""
✓ Mission 1 – Statistiques descriptives
  Bafoussam a le temps moyen le plus long, Douala le plus court.
  Toutes les distributions présentent une asymétrie positive
  (queue à droite = commandes très tardives qui rallongent la moyenne).
 
✓ Mission 2 – Bayes
  P(retard | vendredi soir, Douala) = {P_A_given_B*100:.1f}%
  vs taux de base à Douala = {P_A*100:.1f}% → multiplicateur x{risk_ratio:.1f}
 
✓ Mission 3 – Test t de Student
  t = {t_stat:.3f}, p = {p_value:.4f} < 0.05
  → Différence SIGNIFICATIVE entre Douala et Yaoundé.
  Cohen's d = {cohens_d:.3f} → effet {d_interp}.
 
✓ Mission 4 – Corrélations
  La corrélation la plus forte est heure_commande ↔ delivery_time.
  Montant et fidélité ne corrèlent pas significativement avec le temps.
 
✓ Mission 5 – Outliers IQR
  ~{results_iqr['GLOBAL']['% outliers']}% des livraisons globales sont hors des limites IQR.
  Ces valeurs extrêmes hautes correspondent aux créneaux nocturnes
  et aux zones périphériques (Bafoussam).
""")


RÉSUMÉ SEMAINE 3

✓ Mission 1 – Statistiques descriptives
  Bafoussam a le temps moyen le plus long, Douala le plus court.
  Toutes les distributions présentent une asymétrie positive
  (queue à droite = commandes très tardives qui rallongent la moyenne).

✓ Mission 2 – Bayes
  P(retard | vendredi soir, Douala) = 25.0%
  vs taux de base à Douala = 22.4% → multiplicateur x1.1

✓ Mission 3 – Test t de Student
  t = -10.968, p = 0.0000 < 0.05
  → Différence SIGNIFICATIVE entre Douala et Yaoundé.
  Cohen's d = -0.253 → effet modéré.

✓ Mission 4 – Corrélations
  La corrélation la plus forte est heure_commande ↔ delivery_time.
  Montant et fidélité ne corrèlent pas significativement avec le temps.

✓ Mission 5 – Outliers IQR
  ~7.96% des livraisons globales sont hors des limites IQR.
  Ces valeurs extrêmes hautes correspondent aux créneaux nocturnes
  et aux zones périphériques (Bafoussam).

